# Lamplighter — training a GAN

Build **two** models — a generator and a discriminator — wire them together, and
train them in tandem, all from the browser. This notebook does setup and gets the
results back; training runs *in this kernel*, executing exactly the `train()` the
Training tab shows.

The loop:

1. **Start** a session here → 2. build a **Generator** and a **Discriminator** in
   the **Models** view and link them → 3. register MNIST images with
   `sess.data(X=X)` (no labels — a GAN learns the data distribution itself) →
   4. pick the **GAN** recipe in the **Training** tab and press **▶ Run** →
   5. pull the trained `sess.models["generator"]` back here and sample from it.

> This notebook lives in `examples/`; the first cell puts the repo root on
> `sys.path` so `import lamplighter` resolves.

## 1. Start a session

The first run builds the frontend, then serves the app and opens it in your
browser. Subsequent runs are instant.

In [ ]:
import sys
from pathlib import Path

# The repo root (this notebook runs with examples/ as its cwd).
sys.path.insert(0, str(Path.cwd().parent.resolve()))

import lamplighter

sess = lamplighter.start()
sess.url

## 2. Build the two models

Open the **Models** tab — the high-level view where each model is a node you can
arrange, open, and connect. It starts with one model; use it as the
**Generator**, and add a second for the **Discriminator** with the **＋** in the
sidebar. Double-click a model's name in the sidebar to rename it; the **›** button
(or double-clicking its node) opens its canvas.

**Generator** (noise → image):

> Input **Shape** `1, 100` → Linear `256` → LeakyReLU → Linear `512` → LeakyReLU →
> Linear `784` → Tanh → Output

**Discriminator** (image → one real/fake logit):

> Input **Shape** `1, 784` → Linear `512` → LeakyReLU → Linear `256` → LeakyReLU →
> Linear `1` → Output

On every **LeakyReLU**, set **Negative Slope** to `0.2` (the GAN standard — it
trains noticeably better than the `0.01` default).

Back in the **Models** view, drag from the Generator node's right handle to the
Discriminator's left to **link** them. The edge reads `Generator →
Discriminator: N × 784` — confirming the shapes line up (the discriminator sees
the generator's 784-pixel output). The `100` on the generator's Input is the
latent-noise size; the GAN recipe draws `torch.randn(N, 100)` from it, so no
special node is needed.

## 3. Register MNIST — images only

A GAN learns to *generate* data, so it needs no labels: register just the images.
Normalize them to `[-1, 1]` to match the generator's `Tanh` output range.

In [ ]:
import torch
from torchvision import datasets

mnist = datasets.MNIST(root="./data", train=True, download=True)
# Flatten 28x28 -> 784 and scale [0, 255] -> [-1, 1] (matches the Tanh output).
X = (mnist.data.float() / 127.5 - 1.0).view(-1, 784)
torch.manual_seed(0)
X = X[torch.randperm(len(X))[:8000]]  # subsample for a snappy CPU demo

sess.data(X=X)  # images only — no labels for a GAN

## 4. Pick the data

In the **Data** tab, keep the source on **memory** and pick `X` under
**Input(s)** (hit **↻ refresh** first if it's not listed). Because you've selected
the **GAN** recipe's data contract, the Data tab hides the **Target(s)** picker
and the **Validation Split** — a GAN trains on the images alone.

## 5. Train the GAN

Switch to the **Training** tab and choose the **GAN (adversarial)** recipe. Assign
the **Generator** and **Discriminator** roles to your two models (each gets its
own **Learning Rate** — `0.0002` is the classic default).

Set **Epochs** — and give a GAN room to train. A few dozen epochs is just blurry
blobs; **~250** (about **80 seconds** on CPU for this 8k-image subset) is where
legible digits emerge. Then press **▶ Run**.

Two curves stream in below the code: `d_loss` (the discriminator telling real from
fake) and `g_loss` (how well the generator fools it). Watch for them to *settle*
(around `1.0` and `1.4`) rather than one running away — that equilibrium is the
generator and discriminator holding each other in check, and it's when the
samples start to look like digits. If `g_loss` climbs off toward 3+ and stays
there, the discriminator is winning; train longer or lower its learning rate.

In [ ]:
# The run's per-epoch losses, and the two trained models (keyed by role).
print("epochs trained:", len(sess.history["g_loss"]))
generator = sess.models["generator"]
discriminator = sess.models["discriminator"]
generator

## 6. Sample from the generator

Draw random noise, push it through the trained generator, and look at what it
paints — a grid of 28×28 samples. A well-trained run shows a mix: some clean
digits, some ambiguous. If they're uniform blobs or static, it hasn't trained
enough yet (see the epoch note above).

In [ ]:
import matplotlib.pyplot as plt
import torch.nn as nn

generator = generator.to("cpu").eval()
# Latent size = the generator's first linear input (whatever you built it with).
latent = next(m.in_features for m in generator.modules() if isinstance(m, nn.Linear))
with torch.no_grad():
    noise = torch.randn(16, latent)
    fakes = generator(noise).reshape(-1, 28, 28)   # (16, 28, 28), each pixel in [-1, 1]

fig, axes = plt.subplots(2, 8, figsize=(10, 2.7))
for ax, img in zip(axes.ravel(), fakes):
    ax.imshow(img, cmap="gray", vmin=-1, vmax=1)  # [-1, 1] → black..white
    ax.axis("off")
fig.suptitle(f"generated digits (from {latent}-dim noise)")
plt.tight_layout()
plt.show()

## 7. Keep both models

The two models save together. `sess.checkpoint("mnist-gan")` keeps the run in the
app's **Checkpoints** strip (and you can **▶ Resume** it toward more epochs);
**⬇ Weights** — or `sess.save_checkpoint(path)` — writes one self-contained `.pt`
holding *both* models. Reload a single model from it by role:

In [ ]:
sess.save_checkpoint("mnist-gan.pt")

# A multi-model checkpoint holds every model; pick one by its role name.
generator2, snapshot = lamplighter.load_checkpoint("mnist-gan.pt", model="generator")
with torch.no_grad():
    identical = torch.equal(generator2(noise), generator(noise))
print("reloaded generator matches:", identical)  # same weights, same output

## 8. Tear down

Stops the server thread. (A kernel restart also stops it.)

In [ ]:
lamplighter.stop()